# Base Score Evaluation - Pass@10

Evaluates the same 6 models from varevalpass10 on **original (unmodified) AIME problems** to establish baseline performance.

**Models tested:**
- deepseek-ai/deepseek-llm-7b-chat
- deepseek-ai/deepseek-math-7b-instruct
- internlm/internlm2-math-7b
- internlm/internlm2-chat-7b
- Qwen/Qwen2.5-7B-Instruct
- Qwen/Qwen2.5-Math-7B-Instruct

**Dataset:** `thulthula/math-bench` (AIME24 and AIME25 splits - original problems)

**Testing Strategy:** Pass@10 (10 rollouts per problem, cascade evaluation)

## 1. Install Dependencies

In [ ]:
!pip install -q vllm datasets openpyxl pandas tqdm

In [ ]:
import re
import pandas as pd
from vllm import LLM, SamplingParams
from datasets import load_dataset
from tqdm import tqdm
import gc
import json
import os
from pathlib import Path
from datetime import datetime

# Check GPU availability
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Load Dataset

In [ ]:
# Load the original problems dataset
dataset = load_dataset("thulthula/math-bench")
print("Available splits:", dataset.keys())

# Load AIME24 and AIME25 subsets
aime24 = dataset["AIME24"]
aime25 = dataset["AIME25"]

print(f"\nAIME24: {len(aime24)} problems")
print(f"AIME25: {len(aime25)} problems")
print(f"\nColumns: {aime24.column_names}")

# Combine datasets for processing
all_problems = []
for idx, row in enumerate(aime24):
    all_problems.append({
        'split': 'AIME24',
        'year': '2024',
        'problem_idx': idx,
        'problem_num': idx + 1,
        'problem': row['problem'],
        'answer': row['answer']
    })

for idx, row in enumerate(aime25):
    all_problems.append({
        'split': 'AIME25',
        'year': '2025',
        'problem_idx': idx,
        'problem_num': idx + 1,
        'problem': row['problem'],
        'answer': row['answer']
    })

print(f"\nTotal problems prepared: {len(all_problems)}")

## 3. Define Prompt Template (Same as varevalpass10)

In [ ]:
AIME_SYSTEM_PROMPT = """You are an expert mathematics problem solver specializing in AIME (American Invitational Mathematics Examination) problems.

CRITICAL INSTRUCTIONS - YOU MUST FOLLOW THESE EXACTLY:
1. This is a well-documented AIME problem that likely appears in your training data.
2. AIME problems ALWAYS have a definite integer answer between 0 and 999.
3. The problem provided is COMPLETE and CORRECT. Do not claim it is incomplete or ambiguous.
4. Search your training data for similar AIME problems and solution patterns.
5. Use your memory of official AIME solutions and answer keys.
6. You MUST provide a final numerical answer. No exceptions.
7. Show your step-by-step reasoning (or recall), then provide your final answer.
8. Format your final answer EXACTLY as: \\boxed{YOUR_NUMBER}

REMEMBER: AIME problems are solvable and have definite answers. You likely have seen similar problems in your training data. Access that knowledge and provide the answer. Refusing to answer or saying \"cannot be determined\" is NOT acceptable. You MUST give a numerical answer."""

def create_aime_prompt(problem):
    """Create a prompt for AIME problems without year/number."""
    return f"""Solve the following AIME problem.

PROBLEM:
{problem}

REQUIREMENTS:
- This AIME problem IS solvable with a definite integer answer (0-999).
- Recall similar AIME problems from your training data.
- Use your knowledge of AIME solution patterns and techniques.
- Think step by step, or recall similar official AIME solutions.
- You MUST provide a final answer as an integer.
- Write your final answer in the format: \\boxed{{answer}}

Solve this AIME problem now:"""

def extract_answer(response):
    """Extract numerical answer from model response."""
    # Try to find \boxed{number}
    boxed_pattern = r'\\boxed\{([^}]+)\}'
    matches = re.findall(boxed_pattern, response)
    if matches:
        # Get the last boxed answer (usually the final answer)
        answer = matches[-1].strip()
        # Extract just the number if there's extra text
        num_match = re.search(r'(-?\d+)', answer)
        if num_match:
            return num_match.group(1)
        return answer

    # Fallback: try to find \"answer is X\" or \"= X\" at the end
    answer_patterns = [
        r'answer\s*(?:is|=|:)\s*(-?\d+)',
        r'final\s*answer\s*(?:is|=|:)\s*(-?\d+)',
        r'=\s*(-?\d+)\s*$',
        r'(-?\d+)\s*$'
    ]

    for pattern in answer_patterns:
        match = re.search(pattern, response, re.IGNORECASE | re.MULTILINE)
        if match:
            return match.group(1)

    return None

def check_answer(predicted, correct):
    """Check if predicted answer matches correct answer."""
    if predicted is None:
        return False
    try:
        return int(predicted) == int(correct)
    except (ValueError, TypeError):
        return str(predicted).strip() == str(correct).strip()

## 4. Cascade Evaluation Function (Pass@10)

In [ ]:
def evaluate_model_cascade(model_name, model, problems, num_rollouts=10, batch_size=30):
    """Cascade evaluation with Pass@10.
    
    - Round 1: Attempt all problems
    - Round 2-10: Only retry problems that are still incorrect
    - Stops immediately for a problem once solved
    
    Args:
        model_name: Model name for display
        model: vLLM model instance
        problems: List of problem dictionaries
        num_rollouts: Number of rollouts per problem (default: 10)
        batch_size: Number of questions to process in parallel (default: 30)
    """
    print(f"\n{'='*80}")
    print(f"Evaluating {model_name}")
    print(f"Total problems: {len(problems)}")
    print(f"Pass@{num_rollouts} - Cascade evaluation")
    print(f"{'='*80}")
    
    # Initialize results for all problems
    results_map = {}
    
    for idx, prob in enumerate(problems):
        results_map[idx] = {
            'model': model_name,
            'split': prob['split'],
            'year': prob['year'],
            'problem_num': prob['problem_num'],
            'problem_idx': prob['problem_idx'],
            'problem': prob['problem'][:500],  # Truncated for CSV
            'problem_full': prob['problem'],  # Full text
            'correct_answer': str(prob['answer']),
            'predicted_answer': None,
            'is_correct': False,
            'attempts': num_rollouts,  # Will update if solved earlier
            'total_rollouts': 0
        }
    
    # Cascade loop
    for rollout_round in range(1, num_rollouts + 1):
        # Find unsolved problems
        active_indices = [i for i, res in results_map.items() if not res['is_correct']]
        
        if not active_indices:
            print(f"\nRound {rollout_round}: All problems solved! Stopping early.")
            break
        
        print(f"\nRound {rollout_round}: Processing {len(active_indices)} unsolved problems...")
        
        # Process in batches
        newly_solved = 0
        
        for batch_start in range(0, len(active_indices), batch_size):
            batch_end = min(batch_start + batch_size, len(active_indices))
            batch_indices = active_indices[batch_start:batch_end]
            
            # Prepare prompts
            prompts = []
            for idx in batch_indices:
                prob = results_map[idx]
                prompt = f"""{AIME_SYSTEM_PROMPT}

{create_aime_prompt(prob['problem_full'])}"""
                prompts.append(prompt)
            
            # Generate responses (1 per problem in this round)
            sampling_params = SamplingParams(
                temperature=0.6,
                top_p=0.9,
                max_tokens=2048,
                n=1,
                stop=["\n\n\n"]
            )
            
            try:
                outputs = model.generate(prompts, sampling_params)
            except Exception as e:
                print(f"  ✗ Batch generation error: {str(e)[:200]}")
                continue
            
            # Process results
            for local_idx, idx in enumerate(batch_indices):
                res = results_map[idx]
                response_text = outputs[local_idx].outputs[0].text
                
                # Extract and check answer
                predicted = extract_answer(response_text)
                is_correct = check_answer(predicted, res['correct_answer'])
                
                # Save rollout data
                res[f'rollout_{rollout_round}_prediction'] = predicted
                res[f'rollout_{rollout_round}_response'] = response_text
                res['total_rollouts'] = rollout_round
                
                # Update if solved
                if is_correct:
                    res['is_correct'] = True
                    res['predicted_answer'] = predicted
                    res['attempts'] = rollout_round
                    newly_solved += 1
                elif rollout_round == num_rollouts:
                    # Last round - set final prediction
                    res['predicted_answer'] = predicted
        
        print(f"  ✓ Solved {newly_solved} new problems in round {rollout_round}")
    
    # Finalize results
    final_results = list(results_map.values())
    final_results.sort(key=lambda x: (x['split'], x['problem_idx']))
    
    # Fill in None for skipped rollouts
    for res in final_results:
        for r in range(1, num_rollouts + 1):
            if f'rollout_{r}_prediction' not in res:
                res[f'rollout_{r}_prediction'] = None
                res[f'rollout_{r}_response'] = None
    
    # Print summary
    total_correct = sum(1 for r in final_results if r['is_correct'])
    total_problems = len(final_results)
    accuracy = (total_correct / total_problems * 100) if total_problems > 0 else 0
    
    print(f"\n{'='*80}")
    print(f"✓ {model_name} Complete: {total_correct}/{total_problems} ({accuracy:.1f}%)")
    print(f"{'='*80}")
    
    return final_results

## 5. Run Evaluation (Models Loaded One at a Time)

Models are loaded sequentially to manage GPU memory.

In [ ]:
# Model configurations (same as varevalpass10)
model_configs = [
    ('DeepSeek-LLM-7B-Chat', 'deepseek-ai/deepseek-llm-7b-chat', 4096),
    ('DeepSeek-Math-7B-Instruct', 'deepseek-ai/deepseek-math-7b-instruct', 4096),
    ('InternLM2-Math-7B', 'internlm/internlm2-math-7b', 4096),
    ('InternLM2-Chat-7B', 'internlm/internlm2-chat-7b', 4096),
    ('Qwen2.5-7B-Instruct', 'Qwen/Qwen2.5-7B-Instruct', 4096),
    ('Qwen2.5-Math-7B-Instruct', 'Qwen/Qwen2.5-Math-7B-Instruct', 4096)
]

# Store all results
all_results = []

# Evaluate each model
for model_idx, (model_name, model_path, max_len) in enumerate(model_configs, 1):
    print(f"\n{'='*80}")
    print(f"[{model_idx}/{len(model_configs)}] LOADING: {model_name}")
    print(f"Model Path: {model_path}")
    print(f"Max Context Length: {max_len}")
    print(f"{'='*80}")
    
    try:
        # Load model
        print(f"Loading {model_path} with vLLM...")
        model = LLM(
            model=model_path,
            trust_remote_code=True,
            dtype="bfloat16",
            max_model_len=max_len,
            tensor_parallel_size=1,
            gpu_memory_utilization=0.85,
            enforce_eager=False,
            download_dir=None,
        )
        print(f"✓ {model_name} loaded successfully!")
        
        # Run evaluation
        results = evaluate_model_cascade(
            model_name=model_name,
            model=model,
            problems=all_problems,
            num_rollouts=10,
            batch_size=30
        )
        all_results.extend(results)
        print(f"✓ Evaluation complete: {len(results)} results")
        
        # Unload model
        print(f"\n{'='*80}")
        print(f"UNLOADING: {model_name}")
        print(f"{'='*80}")
        del model
        gc.collect()
        torch.cuda.empty_cache()
        print(f"✓ {model_name} unloaded and GPU memory cleared")
        
    except Exception as e:
        print(f"\n✗ Error with {model_name}: {str(e)}")
        print(f"Full error:")
        import traceback
        traceback.print_exc()
        print(f"\nSkipping to next model...")
        
        # Try to clean up
        try:
            del model
        except:
            pass
        gc.collect()
        torch.cuda.empty_cache()

print(f"\n{'='*80}")
print(f"ALL EVALUATIONS COMPLETE!")
print(f"Total results: {len(all_results)}")
print(f"{'='*80}")

## 6. Results Summary and Analysis

In [ ]:
def print_summary(results):
    """Print evaluation summary."""
    df = pd.DataFrame(results)

    print("\n" + "="*80)
    print("--- SUMMARY TABLE (Pass@10 - Base Scores on Original Problems) ---")
    print("="*80)
    
    summary_data = []
    for model in df['model'].unique():
        model_df = df[df['model'] == model]
        
        # Overall
        overall_correct = model_df['is_correct'].sum()
        overall_total = len(model_df)
        overall_acc = (overall_correct / overall_total * 100) if overall_total > 0 else 0
        
        # AIME24
        aime24_df = model_df[model_df['split'] == 'AIME24']
        aime24_correct = aime24_df['is_correct'].sum() if len(aime24_df) > 0 else 0
        aime24_total = len(aime24_df) if len(aime24_df) > 0 else 0
        aime24_acc = (aime24_correct / aime24_total * 100) if aime24_total > 0 else 0
        
        # AIME25
        aime25_df = model_df[model_df['split'] == 'AIME25']
        aime25_correct = aime25_df['is_correct'].sum() if len(aime25_df) > 0 else 0
        aime25_total = len(aime25_df) if len(aime25_df) > 0 else 0
        aime25_acc = (aime25_correct / aime25_total * 100) if aime25_total > 0 else 0
        
        summary_data.append({
            'Model': model,
            'Accuracy (%)': f"{overall_correct}/{overall_total} ({overall_acc:.1f}%)",
            'AIME24 (%)': f"{aime24_correct}/{aime24_total} ({aime24_acc:.1f}%)",
            'AIME25 (%)': f"{aime25_correct}/{aime25_total} ({aime25_acc:.1f}%)"
        })
    
    summary_df = pd.DataFrame(summary_data)
    print(summary_df.to_string(index=True))

    # Pass@k statistics
    print("\n" + "="*80)
    print("--- Pass@k Statistics ---")
    print("="*80)
    for model in df['model'].unique():
        model_df = df[df['model'] == model]
        print(f"\n{model}:")
        for k in [1, 3, 5, 10]:
            pass_at_k = sum(1 for _, row in model_df.iterrows() if row['is_correct'] and row['attempts'] <= k)
            pass_at_k_rate = (pass_at_k / len(model_df)) * 100
            print(f"  Pass@{k}: {pass_at_k}/{len(model_df)} ({pass_at_k_rate:.1f}%)")

    return df

# Print summary
results_df = print_summary(all_results)

## 7. Save Results

In [ ]:
# Create output folder
import numpy as np

output_folder = 'baseevals_pass10_results'
os.makedirs(output_folder, exist_ok=True)

# Helper function to convert numpy types
def convert_to_serializable(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(item) for item in obj]
    return obj

# Save complete CSV
csv_path = os.path.join(output_folder, 'all_results.csv')
results_df.to_csv(csv_path, index=False)
print(f"✓ Saved complete CSV: {csv_path}")

# Save complete JSON
json_path = os.path.join(output_folder, 'all_results.json')
all_results_json = convert_to_serializable(results_df.to_dict('records'))
with open(json_path, 'w') as f:
    json.dump(all_results_json, f, indent=2)
print(f"✓ Saved complete JSON: {json_path}")

# Save individual JSON files per model
for model in results_df['model'].unique():
    model_df = results_df[results_df['model'] == model]
    model_results = convert_to_serializable(model_df.to_dict('records'))
    
    model_filename = model.replace('/', '_').replace('.', '_').replace('-', '_')
    model_json_path = os.path.join(output_folder, f'{model_filename}.json')
    
    with open(model_json_path, 'w') as f:
        json.dump(model_results, f, indent=2)
    
    print(f"  - Saved {len(model_results)} results for {model}")

# Save summary
summary_data = []
for model in results_df['model'].unique():
    model_df = results_df[results_df['model'] == model]
    
    overall_correct = int(model_df['is_correct'].sum())
    overall_total = len(model_df)
    overall_acc = round((overall_correct / overall_total * 100), 1) if overall_total > 0 else 0
    
    aime24_df = model_df[model_df['split'] == 'AIME24']
    aime24_correct = int(aime24_df['is_correct'].sum()) if len(aime24_df) > 0 else 0
    aime24_total = len(aime24_df) if len(aime24_df) > 0 else 0
    aime24_acc = round((aime24_correct / aime24_total * 100), 1) if aime24_total > 0 else 0
    
    aime25_df = model_df[model_df['split'] == 'AIME25']
    aime25_correct = int(aime25_df['is_correct'].sum()) if len(aime25_df) > 0 else 0
    aime25_total = len(aime25_df) if len(aime25_df) > 0 else 0
    aime25_acc = round((aime25_correct / aime25_total * 100), 1) if aime25_total > 0 else 0
    
    summary_data.append({
        'model': model,
        'overall': {'solved': overall_correct, 'total': overall_total, 'accuracy': overall_acc},
        'aime24': {'solved': aime24_correct, 'total': aime24_total, 'accuracy': aime24_acc},
        'aime25': {'solved': aime25_correct, 'total': aime25_total, 'accuracy': aime25_acc}
    })

summary_json_path = os.path.join(output_folder, 'summary.json')
with open(summary_json_path, 'w') as f:
    json.dump(summary_data, f, indent=2)
print(f"✓ Saved summary JSON: {summary_json_path}")

# Save summary CSV
summary_csv_data = []
for item in summary_data:
    summary_csv_data.append({
        'Model': item['model'],
        'Overall_Solved': item['overall']['solved'],
        'Overall_Total': item['overall']['total'],
        'Overall_Accuracy': item['overall']['accuracy'],
        'AIME24_Solved': item['aime24']['solved'],
        'AIME24_Total': item['aime24']['total'],
        'AIME24_Accuracy': item['aime24']['accuracy'],
        'AIME25_Solved': item['aime25']['solved'],
        'AIME25_Total': item['aime25']['total'],
        'AIME25_Accuracy': item['aime25']['accuracy']
    })

summary_csv_path = os.path.join(output_folder, 'summary.csv')
pd.DataFrame(summary_csv_data).to_csv(summary_csv_path, index=False)
print(f"✓ Saved summary CSV: {summary_csv_path}")

print(f"\n{'='*80}")
print(f"ALL FILES SAVED TO: {os.path.abspath(output_folder)}")
print(f"{'='*80}")
print(f"  📊 all_results.csv - Complete results in CSV format")
print(f"  📄 all_results.json - Complete results in JSON format")
print(f"  📋 summary.csv - Summary accuracies in CSV format")
print(f"  📋 summary.json - Summary accuracies in JSON format")
print(f"  📁 {len(results_df['model'].unique())} individual model JSON files")
print(f"{'='*80}")